# Treinamento dos Modelos


Este notebook realiza o treinamento de modelos de risco de crédito e registra os modelos treinados no Unity Catalog para gerenciamento centralizado.

In [0]:
%pip install databricks-sdk==0.36.0 mlflow==2.19.0 databricks-feature-store==0.17.0 scikit-learn==1.5.2
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
import plotly.express as px
from datetime import datetime

# ============================================================
# Preencha apenas estas variáveis (mesmo padrão do 01-Criacao-dados)
# ============================================================
CATALOGO = ""
SCHEMA = ""
PREFIXO = ""

identificador_schema = f"{CATALOGO}.{SCHEMA}"

# Criando tabela de treino

In [0]:
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()
features_set = fs.read_table(name=f"{identificador_schema}.{PREFIXO}credit_decisioning_features")
display(features_set)

In [0]:
# Criando a coluna de label "defaulted": 1 se atraso > 60 dias, senão 0
credit_bureau_label = (
    spark.table(f"{identificador_schema}.{PREFIXO}credit_bureau_gold")
         .withColumn("defaulted", F.when(col("CREDIT_DAY_OVERDUE") > 60, 1)
                                        .otherwise(0))
         .select("cust_id", "defaulted")
)

# Como podemos ver, temos um dataset bastante desbalanceado
df = credit_bureau_label.groupBy('defaulted').count().toPandas()
px.pie(df, values='count', names='defaulted', title='Proporção de inadimplência')

In [0]:
training_dataset = credit_bureau_label.join(features_set, "cust_id", "inner")

## Balanceamento do dataset

Vamos realizar downsample e upsample no dataset para melhorar a performance do modelo.

In [0]:
major_df = training_dataset.filter(col("defaulted") == 0)
minor_df = training_dataset.filter(col("defaulted") == 1)

# Duplicar as linhas da classe minoritária
oversampled_df = minor_df.union(minor_df)

# Reduzir a classe majoritária
undersampled_df = major_df.sample(oversampled_df.count() / major_df.count() * 3, 42)

# Combinar ambas
train_df = undersampled_df.unionAll(oversampled_df).drop('cust_id').na.fill(0)

# Salvar como tabela para uso pelo AutoML
tabela_treino = f"{identificador_schema}.{PREFIXO}credit_risk_train_df"
train_df.write.mode('overwrite').saveAsTable(tabela_treino)
train_df = spark.table(tabela_treino)

# Visualizar a proporção de inadimplência após balanceamento
px.pie(train_df.groupBy('defaulted').count().toPandas(), values='count', names='defaulted', title='Proporção de inadimplência (após balanceamento)')

# Treinamento

In [0]:
from databricks import feature_store
import mlflow
from mlflow import MlflowClient
from pyspark.sql.functions import col
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mlflow.set_registry_uri('databricks-uc')
mlflow.set_experiment(f"/{PREFIXO}_treinamento")

In [0]:
# Carregar dados da tabela credit_risk_train_df
credit_risk_pdf = spark.table(f"{identificador_schema}.{PREFIXO}credit_risk_train_df").toPandas()

# Separar features e target (defaulted)
X_cr = credit_risk_pdf.drop(columns=["defaulted"])
y_cr = credit_risk_pdf["defaulted"]

# Split treino/teste
X_train_cr, X_test_cr, y_train_cr, y_test_cr = train_test_split(X_cr, y_cr, test_size=0.2, random_state=42)
print(f"Treino: {X_train_cr.shape[0]} | Teste: {X_test_cr.shape[0]}")
print(f"Target 'defaulted' - proporção positivos: {y_cr.mean():.2%}\n")

# Mesmos 5 modelos
models_cr = {
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "AdaBoost": AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "ElasticNet": ElasticNet(alpha=0.5, l1_ratio=0.5, random_state=42, max_iter=1000)
}

results_cr = []

for name, m in models_cr.items():
    with mlflow.start_run(run_name=f"{name}_credit_risk") as run:
        m.fit(X_train_cr, y_train_cr)
        y_pred_cr = m.predict(X_test_cr)

        rmse_cr = np.sqrt(mean_squared_error(y_test_cr, y_pred_cr))
        mae_cr = mean_absolute_error(y_test_cr, y_pred_cr)
        r2_cr = r2_score(y_test_cr, y_pred_cr)

        mlflow.log_params(m.get_params())
        mlflow.log_metric("rmse", rmse_cr)
        mlflow.log_metric("mae", mae_cr)
        mlflow.log_metric("r2_score", r2_cr)
        mlflow.sklearn.log_model(m, "model", input_example=X_test_cr.iloc[:5])

        # Guardar run_id para registro posterior sem novo experimento
        results_cr.append({"Model": name, "RMSE": round(rmse_cr, 4), "MAE": round(mae_cr, 4), "R²": round(r2_cr, 4), "run_id": run.info.run_id})
        print(f"{name}: RMSE={rmse_cr:.4f}, MAE={mae_cr:.4f}, R²={r2_cr:.4f}")

print("\n--- Comparação (credit_risk_train_df) ---")
comparison_cr_df = pd.DataFrame(results_cr).sort_values("R²", ascending=False)
display(comparison_cr_df)

In [0]:
# Selecionar o melhor modelo com base no maior R²
best_model_row = comparison_cr_df.sort_values("R²", ascending=False).iloc[0]
best_model_name = best_model_row["Model"]
print(f"Melhor modelo: {best_model_name} (R²={best_model_row['R²']})")

In [0]:
# Registrar o modelo no Unity Catalog
model_name = f"{PREFIXO}_credit_regression"
catalog = CATALOGO
db = SCHEMA

# Recuperar o run_id do melhor modelo já logado na célula 13 — sem novo treino
best_run_id = comparison_cr_df.loc[comparison_cr_df["Model"] == best_model_name, "run_id"].values[0]

client = MlflowClient()
registered_model = mlflow.register_model(f'runs:/{best_run_id}/model', f"{catalog}.{db}.{model_name}")
client.set_registered_model_alias(name=f"{catalog}.{db}.{model_name}", alias="prod", version=registered_model.version)

print(f"Modelo registrado: {catalog}.{db}.{model_name} v{registered_model.version} (alias: prod)")